In [46]:
import asyncio
import operator
import json
from typing import Annotated, Sequence, TypedDict

from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode

In [2]:
EMBEDDING_MODEL = 'nomic-embed-text:latest'
LOCAL_LLM = 'gemma4:e4b'
TEMPERATURE = 0.7

In [3]:
embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

In [4]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    urls = [f'https://en.wikivoyage.org/wiki/{destination}' for destination in destinations]
    loader = AsyncHtmlLoader(urls)
    print("Downloading destination pages ...")
    docs = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
    chunks = sum([splitter.split_documents([d]) for d in docs], [])

    print(f"Embedding {len(chunks)} chunks ...")
    BATCH_SIZE = 100  # Safe batch size for Ollama requests
    
    vectordb_client = Chroma.from_documents(
        documents=chunks[:BATCH_SIZE],
        embedding=embedding,
    )
    
    for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        print(f"Processing batch {i} to {min(i + BATCH_SIZE, len(chunks))}...")
        vectordb_client.add_documents(batch)
        
    print("Vector store ready.\n")
    return vectordb_client

In [5]:
UK_DESTINATIONS = [
    'Cornwall',
    'North_Cornwall',
    'South_Cornwall',
    'West_Cornwall',
    'Truro_(England)',
    'Newquay',
    'Port_Isaac',
    'St_Ives',
]

async def get_travel_info_vectorstore() -> Chroma:
    vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
    return vectorstore_client

In [6]:
ti_vectorstore_client = await get_travel_info_vectorstore()
ti_retriever = ti_vectorstore_client.as_retriever()

Fetching pages: 100%|##################################################| 4/4 [00:00<00:00, 10.86it/s]


Embedding 1145 chunks ...
Processing batch 100 to 200...
Processing batch 200 to 300...
Processing batch 300 to 400...
Processing batch 400 to 500...
Processing batch 500 to 600...
Processing batch 600 to 700...
Processing batch 700 to 800...
Processing batch 800 to 900...
Processing batch 900 to 1000...
Processing batch 1000 to 1100...
Processing batch 1100 to 1145...
Vector store ready.



In [7]:
@tool
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = ti_retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)

llm_with_tools = llm_model.bind_tools([search_travel_info])    

In [38]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

class ToolsExecutionNode:
    """Execute tools requested by the LLM in the last AIMessage."""

    def __init__(self, tools: Sequence):
        self._tools_by_name = {t.name: t for t in tools}

    def __call__(self, state: dict):
        #print(state)
        messages = state.get("messages", [])
        tool_calls = getattr(messages[-1], 'tool_calls', []) if messages else []
        tool_messages = []
        for tool_call in tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool = self._tools_by_name[tool_name]
            result = tool.invoke(tool_args)
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    name=tool_name,
                    tool_call_id=tool_call['id'],
                )
            )
        return {"messages": tool_messages}

In [39]:
def llm_node(state: AgentState):
    """LLM node that decides whether 
    to call the search tool."""
    current_messages = state["messages"]
    respose_message = llm_with_tools.invoke(current_messages)
    return {"messages": [respose_message]}

In [47]:
builder = StateGraph(AgentState)
builder.add_node("llm_node", llm_node)
#builder.add_node("tools", ToolsExecutionNode([search_travel_info]))
builder.add_node("tools", ToolNode([search_travel_info]))

builder.add_conditional_edges("llm_node", tools_condition)
builder.add_edge("tools", "llm_node")

builder.set_entry_point("llm_node")
travel_info_agent = builder.compile()

In [48]:
def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        result = travel_info_agent.invoke(state)
        print("\n\n\n")
        print(result)
        print("\n\n\n")
        response_msg = result["messages"][-1].content
        print(f"Assistant: {response_msg}\n")

In [49]:
chat_loop()

UK Travel Assistant (type 'exit' to quit)


You:  I want to see some castles in the Cornish area.






{'messages': [HumanMessage(content='I want to see some castles in the Cornish area.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-06T21:52:30.363751Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3389901209, 'load_duration': 211103834, 'prompt_eval_count': 81, 'prompt_eval_duration': 25195000, 'eval_count': 219, 'eval_duration': 3147744000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e9eec-4d5c-76c2-80c5-3d554c0567ad-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'castles in Cornwall'}, 'id': '46e06706-6a06-4392-9018-9fff50a1d517', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 219, 'total_tokens': 300}), ToolMessage(content='<p id="mwog">Arthur and seat of the kings of Cornwall. Earl Richard of Cornwall and King of the Romans built the present medi

You:  exit


In [ ]:
You:  I want to see some castles in the Cornish area.
"state" in tool function:
{'messages': [HumanMessage(content='I want to see some castles in the Cornish area.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-06T21:13:50.333612Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5221087083, 'load_duration': 3582710708, 'prompt_eval_count': 81, 'prompt_eval_duration': 91661000, 'eval_count': 108, 'eval_duration': 1544559000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e9ec8-df97-7462-840e-23986f9fdb7c-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'castles in Cornish area'}, 'id': 'c6a23efe-9535-4bba-8f41-d7c2619e4865', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 108, 'total_tokens': 189})]}

Result:
{'messages': [HumanMessage(content='I want to see some castles in the Cornish area.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-06T21:13:50.333612Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5221087083, 'load_duration': 3582710708, 'prompt_eval_count': 81, 'prompt_eval_duration': 91661000, 'eval_count': 108, 'eval_duration': 1544559000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e9ec8-df97-7462-840e-23986f9fdb7c-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'castles in Cornish area'}, 'id': 'c6a23efe-9535-4bba-8f41-d7c2619e4865', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 108, 'total_tokens': 189}), ToolMessage(content='"<p id=\\"mwog\\">Arthur and seat of the kings of Cornwall. Earl Richard of Cornwall and King of the Romans built the present medieval castle at the site. Ongoing excavations are revealing a Cornish royal seat of the period 400 to 700 AD.</p>\\n---\\nThe Cornish are extremely proud of their history and heritage, which pre-date the arrival of the <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/Roman_Empire\\" title=\\"Roman Empire\\" id=\\"mwdg\\">Romans</a> or Anglo-Saxons in Britain, and many Cornish people are loyal to their county. You may even see some Cornish people wearing kilts and playing Cornish pipes at cultural and other gatherings. Do not confuse the kilts with Scottish kilts. Cornwall is recognised as a separate nation by many international organisations, including the EU. One such popular organisation is <i id=\\"mwdw\\">Gorsedh Kernow</i>, aimed at promoting Cornish culture and festivals such as the Gorsedd.</p>\\n---\\n<p id=\\"mwEQ\\">Cornwall is a popular destination for those interested in cultural tourism, due to its long association with visual and written arts and its wealth of archaeology. Its <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/Mining\\" title=\\"Mining\\" class=\\"mw-redirect\\" id=\\"mwEg\\">mining</a> heritage has been recognised by the <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/UNESCO_World_Heritage_List#United_Kingdom\\" title=\\"UNESCO World Heritage List\\" id=\\"mwEw\\">United Nations (UNESCO)</a>. Over 30% of the county is designated as an <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/Areas_of_Outstanding_Natural_Beauty\\" title=\\"Areas of Outstanding Natural Beauty\\" class=\\"mw-redirect\\" id=\\"mwFA\\">Area of Outstanding Natural Beauty (AONB)</a>, giving it national status and protection. Cornwall has always been fiercely proud of its Celtic heritage, and for many residents, their Cornish identity supersedes their Englishness or Britishness.</p>\\n---\\n<p id=\\"mwBw\\"><b id=\\"mwCA\\">Cornwall</b> (Cornish: <i id=\\"mwCQ\\">Kernow</i>) is a county in the southwest of England. Lying west of <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/Devon\\" title=\\"Devon\\" id=\\"mwCg\\">Devon</a> from which it is separated by the River Tamar, Cornwall is one of the more isolated and distinctive parts of the <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/United_Kingdom\\" title=\\"United Kingdom\\" id=\\"mwCw\\">United Kingdom</a> but is also one of its most popular with holidaymakers. Its relatively warm climate, long coastline, amazing scenery, and diverse <a rel=\\"mw:WikiLink\\" href=\\"//en.wikivoyage.org/wiki/Celts\\" title=\\"Celts\\" id=\\"mwDA\\">Celtic</a> heritage (combined with tales of smuggling, pirates and King Arthur!) go only part of the way to explaining its appeal.</p>"', name='search_travel_info', tool_call_id='c6a23efe-9535-4bba-8f41-d7c2619e4865'), AIMessage(content='Based on the information retrieved, there is mention of an area associated with "Arthur and seat of the kings of Cornwall," where Earl Richard of Cornwall built a medieval castle. This suggests that historical castles are significant in the region.\n\nWhile I don\'t have a list of specific castles right now, Cornwall has a rich history related to royal seats and folklore (like King Arthur), which often means there are historic sites with castle remnants or actual structures.\n\nTo give you the best recommendations, could you tell me:\n\n1.  **Are you interested in a specific type of experience?** (e.g., ruins, fully intact castles, coastal castles, castles linked to folklore?)\n2.  **Do you have any dates for your trip?** (This helps narrow down which sites are open.)\n\nIn the meantime, I recommend looking up **St Mawes Castle** or **Pendennis Castle**, as these are well-known historical fortifications in Cornwall that might fit what you are looking for!', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-06T21:13:59.2533Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8634549708, 'load_duration': 199085541, 'prompt_eval_count': 846, 'prompt_eval_duration': 416153000, 'eval_count': 546, 'eval_duration': 8014094000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e9ec8-f51a-7e91-9534-88fe2585baee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 846, 'output_tokens': 546, 'total_tokens': 1392})]}




    
You:  Tell me something about York.
[{'name': 'search_travel_info', 'args': {'query': 'York'}, 'id': '4addbb36-5f26-4566-8d99-2283b6607309', 'type': 'tool_call'}]

You:  What can I see in Liverpool?
[{'name': 'search_travel_info', 'args': {'query': 'Liverpool'}, 'id': 'cb1d81b4-49a4-4823-a256-f700b127c714', 'type': 'tool_call'}]

You: What can I see in Northumberland?
content='' additional_kwargs={} response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-05T23:17:30.509411Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3918872458, 'load_duration': 207195292, 'prompt_eval_count': 77, 'prompt_eval_duration': 64241000, 'eval_count': 251, 'eval_duration': 3644318000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'} id='lc_run--019e9a13-c1bd-7280-b8a4-96af6d5f0200-0' tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'What can I see in Northumberland?'}, 'id': 'a216d09f-2970-4b27-ba3f-adf2da15e1b5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 251, 'total_tokens': 328}

You:  Suggest some picturesque hikes in the Lake District.
content='' additional_kwargs={} response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-05T23:06:10.627509Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6142499000, 'load_duration': 2057304375, 'prompt_eval_count': 79, 'prompt_eval_duration': 89872000, 'eval_count': 275, 'eval_duration': 3993993000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'} id='lc_run--019e9a09-5943-70a1-a812-084ab8423243-0' tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'picturesque hikes in the Lake District'}, 'id': '733def4d-78e0-4bc4-a2f6-901d6fb2eda3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 79, 'output_tokens': 275, 'total_tokens': 354}

You:  I want to see some castles in the Cornish area.
content='' additional_kwargs={} response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-05T23:14:56.270112Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7007147166, 'load_duration': 2056932833, 'prompt_eval_count': 81, 'prompt_eval_duration': 90007000, 'eval_count': 335, 'eval_duration': 4858985000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'} id='lc_run--019e9a11-5b2d-7170-bb50-74ab5d8e9414-0' tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'castles in Cornwall'}, 'id': 'f46cd3b9-f886-43d1-b632-4892f4557805', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 81, 'output_tokens': 335, 'total_tokens': 416}


    
[{'name': 'search_travel_info', 'args': {'query': 'castles in Cornwall'}, 'id': 'cf0af30c-eb5f-4880-b992-8a53b4d07523', 'type': 'tool_call'}]


Where are the Hebrides islands?
[]

Who is the current monarch of the UK?
[]

You:  What is the capital of England?
